# Tema 3 (Laboratorul 4) — Rețele Bayesiene (`pgmpy`)

---
## Exercițiul 1 — Problema Monty Hall și implementarea din documentația `pgmpy`

### Explicarea problemei
În celebrul joc **Monty Hall**, un concurent are în față 3 uși închise ($0, 1, 2$): în spatele uneia se află o mașină (premiul), iar în spatele celorlalte două se află capre.
1. Concurentul alege inițial o ușă $P \in \{0, 1, 2\}$, independent de poziția mașinii $C \in \{0, 1, 2\}$, fiecare ușă având probabilitate apriori $\frac{1}{3}$.
2. Prezentatorul (Monty), care **știe** unde este mașina, deschide întotdeauna o altă ușă $H \in \{0, 1, 2\}$ astfel încât:
   * $H \neq P$ (nu deschide ușa aleasă de concurent);
   * $H \neq C$ (nu deschide ușa cu mașina — dezvăluie mereu o capră).
   Dacă are de ales între două uși cu capre (când $P = C$), alege una dintre ele cu probabilitate $\frac{1}{2}$ fiecare.
3. Concurentul este întrebat dacă dorește să își schimbe alegerea către cealaltă ușă rămasă închisă.

### Explicarea structurii rețelei și a codului `pgmpy`
* **Structura grafului (DAG):** `C -> H <- P`. Nodul $H$ este un **collider** (*V-structure*): inițial $C$ și $P$ sunt independente ($C \perp\!\!\!\perp P$), dar imediat ce observăm ușa deschisă de prezentator ($H$), între $C$ și $P$ apare o dependență informațională (*explaining away*).
* **Tabelele CPD (`TabularCPD`):**
  * `cpd_c` și `cpd_p`: distribuții uniforme $\left[\frac{1}{3}, \frac{1}{3}, \frac{1}{3}\right]^T$ pe cele 3 uși.
  * `cpd_h`: o matrice $3 \times 9$ corespunzătoare celor $3 \times 3 = 9$ combinații $(C, P)$:
    * Când $C = P$ (ex. $C=0, P=0$), prezentatorul are două uși cu capre disponibile ($1$ și $2$), deci $P(H=0)=0, P(H=1)=0.5, P(H=2)=0.5$.
    * Când $C \neq P$ (ex. $C=0, P=1$), prezentatorul este obligat să deschidă singura ușă rămasă ($H=2$), deci $P(H=2)=1$.
* **Inferența (`VariableElimination`):** Dacă jucătorul alege $P=0$ și prezentatorul deschide $H=2$, interogarea `infer.query(variables=['C'], evidence={'P': 0, 'H': 2})` arată că:
  * $P(C=0 \mid P=0, H=2) = \frac{1}{3}$ (șansa de câștig dacă păstrează ușa);
  * $P(C=1 \mid P=0, H=2) = \frac{2}{3}$ (șansa de câștig dacă **schimbă** ușa).

In [ ]:
# Implementarea problemei Monty Hall din documentația pgmpy
try:
    try:
        from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
    except ImportError:
        from pgmpy.models import BayesianNetwork
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination

    monty_model = BayesianNetwork([('C', 'H'), ('P', 'H')])

    cpd_c = TabularCPD('C', 3, [[1/3], [1/3], [1/3]])
    cpd_p = TabularCPD('P', 3, [[1/3], [1/3], [1/3]])

    # Coloanele corespund perechilor (C, P): (0,0), (0,1), (0,2), (1,0), (1,1), (1,2), (2,0), (2,1), (2,2)
    cpd_h = TabularCPD(
        'H', 3,
        [
            [0.0, 0.0, 0.0, 0.0, 0.5, 1.0, 0.0, 1.0, 0.5],
            [0.5, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.5],
            [0.5, 1.0, 0.0, 1.0, 0.5, 0.0, 0.0, 0.0, 0.0],
        ],
        evidence=['C', 'P'],
        evidence_card=[3, 3]
    )

    monty_model.add_cpds(cpd_c, cpd_p, cpd_h)
    assert monty_model.check_model()

    infer_mh = VariableElimination(monty_model)
    posterior_c = infer_mh.query(variables=['C'], evidence={'P': 0, 'H': 2})
    print("Distribuția a posteriori pentru ușa cu mașina C, știind P=0 și H=2:")
    print(posterior_c)
except ImportError:
    print("pgmpy nu este instalat în acest mediu. Rezultatul teoretic: P(C=0|P=0,H=2) = 1/3, P(C=1|P=0,H=2) = 2/3.")

---
## Exercițiul 2 — Rețeaua Bayesiană pentru diagnostic medical

### Structura rețelei
Variabile binare: Boala pulmonară ($B$), Tuse ($T$), Dificultate de respirație ($D$), Radiografie anormală ($X$).
Arcele grafului orientat aciclic sunt:
$$B \to T, \quad B \to X, \quad B \to D, \quad T \to D.$$

### d) Calculul analitic prin Regula lui Bayes

1. **Punctul b) — Probabilitatea ca pacientul să aibă boala, știind că are tuse ($T=1$) și dificultăți de respirație ($D=1$):**
   Observăm din graf că nodul $X$ (radiografia) nu are descendenți și este condiționat independent de $\{T, D\}$ dat fiind $B$ ($X \perp\!\!\!\perp \{T, D\} \mid B$), deci prin marginalizarea lui $X$ ($\sum_{x \in \{0,1\}} P(X=x \mid B) = 1$):
   $$P(B, T=1, D=1) = P(B) \cdot P(T=1 \mid B) \cdot P(D=1 \mid B, T=1).$$
   Evaluăm pentru $B=1$ și $B=0$:
   * $P(B=1, T=1, D=1) = 0.1 \times 0.8 \times 0.9 = 0.072$.
   * $P(B=0, T=1, D=1) = 0.9 \times 0.3 \times 0.5 = 0.135$.
   * Probabilitatea marginală a simptomelor:
     $$P(T=1, D=1) = 0.072 + 0.135 = 0.207.$$
   * Prin regula lui Bayes:
     $$P(B=1 \mid T=1, D=1) = \frac{P(B=1, T=1, D=1)}{P(T=1, D=1)} = \frac{0.072}{0.207} = \frac{8}{23} \approx 0.347826 \quad (34.78\%).$$

2. **Punctul c) — Probabilitatea ca radiografia să fi fost anormală ($X=1$), știind că pacientul NU are boala ($B=0$) (și păstrând observațiile $T=1, D=1$):**
   Conform proprietății locale Markov în rețeaua Bayesiană, nodul $X$ este condiționat independent de non-descendenții săi $\{T, D\}$ atunci când cunoaștem părintele său $B$:
   $$X \perp\!\!\!\perp \{T, D\} \mid B.$$
   Prin urmare:
   $$P(X=1 \mid B=0, T=1, D=1) = P(X=1 \mid B=0) = 0.10 \quad (10\%).$$

In [ ]:
# Rezolvarea Exercițiului 2 în pgmpy (cu verificare exactă)
try:
    try:
        from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
    except ImportError:
        from pgmpy.models import BayesianNetwork
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination

    med_model = BayesianNetwork([('B', 'T'), ('B', 'X'), ('B', 'D'), ('T', 'D')])

    cpd_B = TabularCPD('B', 2, [[0.9], [0.1]])
    cpd_T = TabularCPD('T', 2, [[0.7, 0.2],
                                [0.3, 0.8]], evidence=['B'], evidence_card=[2])
    cpd_X = TabularCPD('X', 2, [[0.9, 0.1],
                                [0.1, 0.9]], evidence=['B'], evidence_card=[2])
    # Ordinea coloanelor pentru evidence=['B', 'T']: (0,0), (0,1), (1,0), (1,1)
    cpd_D = TabularCPD('D', 2, [[0.9, 0.5, 0.6, 0.1],
                                [0.1, 0.5, 0.4, 0.9]], evidence=['B', 'T'], evidence_card=[2, 2])

    med_model.add_cpds(cpd_B, cpd_T, cpd_X, cpd_D)
    assert med_model.check_model()

    infer_med = VariableElimination(med_model)

    q_b = infer_med.query(variables=['B'], evidence={'T': 1, 'D': 1})
    q_x = infer_med.query(variables=['X'], evidence={'B': 0, 'T': 1, 'D': 1})

    print(f"b) P(B=1 | T=1, D=1)      = {q_b.values[1]:.6f} (exact: 8/23 = {8/23:.6f})")
    print(f"c) P(X=1 | B=0, T=1, D=1) = {q_x.values[1]:.6f} (exact: 0.100000)")
except ImportError:
    p_b1_td = 0.1 * 0.8 * 0.9
    p_b0_td = 0.9 * 0.3 * 0.5
    print(f"b) P(B=1 | T=1, D=1)      = {p_b1_td / (p_b1_td + p_b0_td):.6f} (exact: 8/23)")
    print(f"c) P(X=1 | B=0, T=1, D=1) = {0.1:.6f}")

---
## Exercițiul 3 — Jocul dintre $J_0$ și $J_1$

### Modelarea probabilistă
* $S \in \{0, 1\}$: cine începe jocul ($S=0 \implies J_0$ începe, $S=1 \implies J_1$ începe), cu $P(S=0) = P(S=1) = 0.5$.
* $N \in \{1, 2, 3, 4, 5, 6\}$: rezultatul zarului normal aruncat în runda 1, cu $P(N=n) = \frac{1}{6}$, independent de $S$.
* $M \in \{0, 1, \dots, 12\}$: numărul de steme obținute în runda 2 din $2N$ aruncări ale monedei celuilalt jucător:
  * Dacă $S=0$ ($J_0$ începe), aruncă $J_1$ cu moneda măsluită ($p = \frac{4}{7}$): $M \mid (S=0, N=n) \sim \text{Binomial}\left(2n, \frac{4}{7}\right)$.
  * Dacă $S=1$ ($J_1$ începe), aruncă $J_0$ cu moneda cinstită ($p = \frac{1}{2}$): $M \mid (S=1, N=n) \sim \text{Binomial}\left(2n, \frac{1}{2}\right)$.
* **Regula de câștig:** Jucătorul din prima rundă câștigă dacă $N \ge M$, altfel câștigă jucătorul din a doua rundă ($N < M$).
  * Calculul exact arată că $P(J_0 \text{ câștigă}) \approx 0.4267$ și $P(J_1 \text{ câștigă}) \approx 0.5733$, deci **jucătorul $J_1$ are șanse mai mari de câștig** (datorită monedei sale măsluite care produce mai multe steme atunci când joacă în runda 2, fără a-i afecta zarul când joacă în runda 1).
* **Punctul c) — Cine a început cel mai probabil dacă în runda 2 s-a obținut $M = 1$ stemă?**
  * $P(S=0 \mid M=1) \approx 0.4529$, iar $P(S=1 \mid M=1) \approx 0.5471$.
  * Prin urmare, **cel mai probabil a început jucătorul $J_1$ ($S=1$)**, deoarece atunci când începe $J_1$, în runda a doua aruncă $J_0$ cu moneda cinstită ($p = 0.5$), care are o probabilitate mai mare de a obține un număr mic de steme ($M=1$) decât moneda măsluită a lui $J_1$ ($p = 4/7$).

In [ ]:
import numpy as np
from math import comb

# a) Simularea jocului de 10.000 de ori
rng = np.random.default_rng(42)
N_sim = 10_000

S_sim = rng.integers(0, 2, size=N_sim)      # 0 = J0 începe, 1 = J1 începe
N_zar = rng.integers(1, 7, size=N_sim)      # zar 1..6
p_moneda = np.where(S_sim == 0, 4.0 / 7.0, 0.5)
M_steme = rng.binomial(2 * N_zar, p_moneda)

# Jucătorul din runda 1 câștigă dacă N >= M
primul_castiga = N_zar >= M_steme
j0_castiga = np.where(S_sim == 0, primul_castiga, ~primul_castiga)

print(f"a) Simulare ({N_sim:,} jocuri):")
print(f"   P(J0 câștigă) estimat = {np.mean(j0_castiga):.4f} (exact: 0.4267)")
print(f"   P(J1 câștigă) estimat = {1 - np.mean(j0_castiga):.4f} (exact: 0.5733)")

# b) & c) Definirea rețelei Bayesiene în pgmpy și inferența P(S | M = 1)
try:
    try:
        from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
    except ImportError:
        from pgmpy.models import BayesianNetwork
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination

    game_model = BayesianNetwork([('S', 'M'), ('N', 'M')])

    cpd_S = TabularCPD('S', 2, [[0.5], [0.5]], state_names={'S': ['J0', 'J1']})
    cpd_N = TabularCPD('N', 6, [[1/6]] * 6, state_names={'N': [1, 2, 3, 4, 5, 6]})

    # Matricea CPD pentru M (13 rânduri: 0..12) condiționată de (S, N) (2 * 6 = 12 coloane)
    cpd_M_values = np.zeros((13, 12))
    col = 0
    for s_idx, p in enumerate([4.0 / 7.0, 0.5]):
        for n_val in range(1, 7):
            for m_val in range(0, 2 * n_val + 1):
                cpd_M_values[m_val, col] = comb(2 * n_val, m_val) * (p ** m_val) * ((1 - p) ** (2 * n_val - m_val))
            col += 1

    cpd_M = TabularCPD(
        'M', 13, cpd_M_values.tolist(),
        evidence=['S', 'N'], evidence_card=[2, 6],
        state_names={'M': list(range(13)), 'S': ['J0', 'J1'], 'N': [1, 2, 3, 4, 5, 6]}
    )

    game_model.add_cpds(cpd_S, cpd_N, cpd_M)
    assert game_model.check_model()

    infer_game = VariableElimination(game_model)
    post_S = infer_game.query(variables=['S'], evidence={'M': 1})
    print("\nc) Inferență în pgmpy pentru P(S | M = 1):")
    print(post_S)
    print(f"Concluzie: Cel mai probabil a început jucătorul {'J1' if post_S.values[1] > post_S.values[0] else 'J0'}.")
except ImportError:
    p_m1_s0 = sum((1/6) * comb(2*n, 1) * ((4/7)**1) * ((3/7)**(2*n - 1)) for n in range(1, 7))
    p_m1_s1 = sum((1/6) * comb(2*n, 1) * (0.5**(2*n)) for n in range(1, 7))
    p_s0_m1 = p_m1_s0 / (p_m1_s0 + p_m1_s1)
    p_s1_m1 = p_m1_s1 / (p_m1_s0 + p_m1_s1)
    print(f"\nc) Calcul exact: P(S=J0 | M=1) = {p_s0_m1:.6f}, P(S=J1 | M=1) = {p_s1_m1:.6f} => A început J1.")